In [2]:

# Step 1: Environment Setup & Load Raw Datasets for Cleaning

import os
import pandas as pd
import numpy as np

RAW_DATA_PATH = "../data/raw"
PROCESSED_DATA_PATH = "../data/processed"

# Processed folder create karna agar exist nahi karta
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

# Raw files load karna
customers = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_customers_dataset.csv"))
geolocation = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_geolocation_dataset.csv"))
order_items = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_order_items_dataset.csv"))
payments = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_order_payments_dataset.csv"))
reviews = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_order_reviews_dataset.csv"))
orders = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_orders_dataset.csv"))
products = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_products_dataset.csv"))
sellers = pd.read_csv(os.path.join(RAW_DATA_PATH, "olist_sellers_dataset.csv"))
category_translation = pd.read_csv(os.path.join(RAW_DATA_PATH, "product_category_name_translation.csv"))

print("[SUCCESS] All raw datasets loaded into memory for cleaning pipeline.")

[SUCCESS] All raw datasets loaded into memory for cleaning pipeline.


In [4]:
# Phase 2: Data Cleaning & Quality

# Step 2: Datetime Conversion & Zip Code Zero-Padding

#  Dictionary mapping define karna

datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

# 1. Datetime Type Casting across Target Datasets
datetime_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

for table_name, cols in datetime_columns.items():
    df = datasets[table_name]
    for col in cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    print(f"[CONVERTED] {table_name}: {len(cols)} columns converted to datetime64[ns]")

# 2. Postal Code Standardisation (Pad leading zeros to ensure strict 5-digit strings)
zip_targets = [
    (customers, "customer_zip_code_prefix"),
    (sellers, "seller_zip_code_prefix"),
    (geolocation, "geolocation_zip_code_prefix")
]

for df, col in zip_targets:
    df[col] = df[col].astype(str).str.zfill(5)

print("[STANDARDIZED] Zip codes zero-padded to 5 digits in customers, sellers, and geolocation")

# Verification
print("\n--- Validation Sample ---")
print("Orders dtypes:")
print(orders[datetime_columns["orders"]].dtypes)
print("\nSample Zip Codes:")
print("Customer Zip:", customers["customer_zip_code_prefix"].iloc[0])
print("Seller Zip:  ", sellers["seller_zip_code_prefix"].iloc[0])
print("Geo Zip:     ", geolocation["geolocation_zip_code_prefix"].iloc[0])

[CONVERTED] orders: 5 columns converted to datetime64[ns]
[CONVERTED] order_items: 1 columns converted to datetime64[ns]
[CONVERTED] reviews: 2 columns converted to datetime64[ns]
[STANDARDIZED] Zip codes zero-padded to 5 digits in customers, sellers, and geolocation

--- Validation Sample ---
Orders dtypes:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

Sample Zip Codes:
Customer Zip: 14409
Seller Zip:   13023
Geo Zip:      01037


In [ ]:
# Step 3: Deduplication & Geolocation Aggregation

# 1. Deduplicate Reviews: Sort by answer timestamp and retain the latest entry per review_id
initial_reviews_count = len(reviews)
reviews_clean = (
    reviews.sort_values("review_answer_timestamp")
    .drop_duplicates(subset=["review_id"], keep="last")
    .reset_index(drop=True)
)
dropped_reviews = initial_reviews_count - len(reviews_clean)
print(f"[REVIEWS] Dropped {dropped_reviews} duplicate reviews. Clean count: {len(reviews_clean)}")

# 2. Geolocation Aggregation: Group by zip prefix and compute average coordinates
initial_geo_count = len(geolocation)
geolocation_clean = (
    geolocation.groupby("geolocation_zip_code_prefix")
    .agg({
        "geolocation_lat": "mean",
        "geolocation_lng": "mean",
        "geolocation_city": "first",
        "geolocation_state": "first"
    })
    .reset_index()
)
print(f"[GEOLOCATION] Aggregated {initial_geo_count} raw rows into {len(geolocation_clean)} unique zip code centers.")

# Check uniqueness verification
print(f"Is reviews_clean unique on review_id? {reviews_clean['review_id'].is_unique}")
print(f"Is geolocation_clean unique on zip prefix? {geolocation_clean['geolocation_zip_code_prefix'].is_unique}")

[REVIEWS] Dropped 814 duplicate reviews. Clean count: 98410
[GEOLOCATION] Aggregated 1000163 raw rows into 19015 unique zip code centers.
Is reviews_clean unique on review_id? True
Is geolocation_clean unique on zip prefix? True


In [6]:
# Step 4: Missing Values Imputation & Payments Cleaning

# 1. Products Imputation
initial_prod_nulls = products["product_category_name"].isna().sum()

# Category imputation
products["product_category_name"] = products["product_category_name"].fillna("uncategorized")

# Numeric dimensions & weight imputation via median
dimension_cols = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "product_photos_qty"
]

for col in dimension_cols:
    median_val = products[col].median()
    products[col] = products[col].fillna(median_val)

print(f"[PRODUCTS] Imputed {initial_prod_nulls} missing categories as 'uncategorized'.")
print(f"[PRODUCTS] Remaining missing values in products: {products[dimension_cols].isna().sum().sum()}")

# 2. Payments Zero-Value Filtering
initial_payments_count = len(payments)
payments_clean = payments[payments["payment_value"] > 0].copy().reset_index(drop=True)
dropped_payments = initial_payments_count - len(payments_clean)

print(f"[PAYMENTS] Dropped {dropped_payments} zero-value records. Clean count: {len(payments_clean)}")

[PRODUCTS] Imputed 610 missing categories as 'uncategorized'.
[PRODUCTS] Remaining missing values in products: 0
[PAYMENTS] Dropped 9 zero-value records. Clean count: 103877


In [7]:
# Step 5: Export Clean Datasets 

export_manifest = {
    "customers_cleaned.csv": customers,
    "geolocation_cleaned.csv": geolocation_clean,
    "order_items_cleaned.csv": order_items,
    "payments_cleaned.csv": payments_clean,
    "reviews_cleaned.csv": reviews_clean,
    "orders_cleaned.csv": orders,
    "products_cleaned.csv": products,
    "sellers_cleaned.csv": sellers,
    "category_translation_cleaned.csv": category_translation
}

print("Saving cleaned datasets to data/processed/ ...")
for filename, df in export_manifest.items():
    filepath = os.path.join(PROCESSED_DATA_PATH, filename)
    df.to_csv(filepath, index=False)
    print(f"[SAVED] {filename} ({len(df):,} rows)")

print("\n[SUCCESS] All 9 cleaned datasets exported successfully!")

Saving cleaned datasets to data/processed/ ...
[SAVED] customers_cleaned.csv (99,441 rows)
[SAVED] geolocation_cleaned.csv (19,015 rows)
[SAVED] order_items_cleaned.csv (112,650 rows)
[SAVED] payments_cleaned.csv (103,877 rows)
[SAVED] reviews_cleaned.csv (98,410 rows)
[SAVED] orders_cleaned.csv (99,441 rows)
[SAVED] products_cleaned.csv (32,951 rows)
[SAVED] sellers_cleaned.csv (3,095 rows)
[SAVED] category_translation_cleaned.csv (71 rows)

[SUCCESS] All 9 cleaned datasets exported successfully!
